In [ ]:
import cv2
import torch
import numpy as np
from facenet_pytorch import MTCNN, InceptionResnetV1
import os

# =============================
# 1. THIẾT LẬP DEVICE
# =============================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Sử dụng thiết bị:", device)

# =============================
# 2. KHỞI TẠO MODEL
# =============================
# MTCNN để trích xuất ảnh mẫu (chỉ lấy 1 khuôn mặt rõ nhất)
mtcnn_enroll = MTCNN(keep_all=False, device=device)

# MTCNN để chạy webcam (lấy tất cả các khuôn mặt xuất hiện trong khung hình)
mtcnn_webcam = MTCNN(keep_all=True, device=device) 

# Mô hình FaceNet
facenet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# =============================
# 3. HÀM LẤY EMBEDDING AN TOÀN
# =============================
def get_embedding(img_path):
    try:
        img = cv2.imread(img_path)
        if img is None:
            print(" Không mở được ảnh:", img_path)
            return None

        # Chuyển BGR (OpenCV) sang RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # mtcnn_enroll sẽ tự động crop, resize 160x160 và chuẩn hóa pixel về [-1, 1]
        face_tensor = mtcnn_enroll(img)
        if face_tensor is None:
            print("Không phát hiện được khuôn mặt:", img_path)
            return None

        # Thêm chiều batch: [3, 160, 160] -> [1, 3, 160, 160]
        face_tensor = face_tensor.unsqueeze(0).to(device)
        
        # Extract embedding
        with torch.no_grad(): 
            embedding = facenet(face_tensor)
            
        return embedding.cpu().numpy()[0]

    except Exception as e:
        print("Lỗi extract embedding:", img_path, e)
        return None

# =============================
# 4. LOAD KNOWN FACES
# =============================
# 4. LOAD KNOWN FACES
# =============================
known_embeddings = {}

known_dir = r"D:\xu_li_anh\lab4\train\18-20"  

if not os.path.exists(known_dir):
    print(f"Thư mục không tồn tại: {known_dir}")
else:
    for filename in os.listdir(known_dir):
        if filename.lower().endswith((".jpg", ".png", ".jpeg")):
            path = os.path.join(known_dir, filename)
            emb = get_embedding(path)
            if emb is not None:
                name = os.path.splitext(filename)[0]
                known_embeddings[name] = emb

print("Đã load embeddings của:", list(known_embeddings.keys()))
if len(known_embeddings) == 0:
    raise ValueError(f"Không có embedding nào được load! Kiểm tra lại xem có ảnh trong {known_dir} không và MTCNN có nhận dạng được mặt trong ảnh không.")

# =============================
# 5. COSINE SIMILARITY
# =============================
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# =============================
# 6. WEBCAM REAL-TIME
# =============================
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        print(" Không đọc được dữ liệu từ camera!")
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Phát hiện các khuôn mặt trong khung hình webcam
    boxes, probs = mtcnn_webcam.detect(rgb)

    if boxes is not None:
        for box, prob in zip(boxes, probs):
            # Bỏ qua các nhận diện có độ tin cậy thấp (nhiễu)
            if prob is None or prob < 0.90:
                continue

            x1, y1, x2, y2 = map(int, box)
            
            # Đảm bảo tọa độ khung không vượt ra ngoài kích thước thật của camera
            h_frame, w_frame, _ = frame.shape
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w_frame, x2), min(h_frame, y2)

            face = rgb[y1:y2, x1:x2]

            try:
                # Resize về chuẩn đầu vào của FaceNet
                face = cv2.resize(face, (160, 160))
            except Exception:
                continue

            # QUAN TRỌNG: Chuẩn hóa thủ công về dải [-1, 1] 
            face = face.astype(np.float32)
            face = (face - 127.5) / 128.0

            # Chuyển đổi thành tensor chuẩn bị đưa vào model: shape [1, 3, 160, 160]
            face_tensor = torch.tensor(face).permute(2, 0, 1).unsqueeze(0).to(device)
            
            with torch.no_grad(): # Tắt gradient để tăng tốc độ khung hình (FPS)
                emb = facenet(face_tensor).cpu().numpy()[0]

            # So sánh với tất cả known embeddings
            sims = [cosine_similarity(emb, k_emb) for k_emb in known_embeddings.values()]
            if len(sims) > 0:
                max_sim = max(sims)
                if max_sim > 0.7:  # Ngưỡng (Threshold): Có thể chỉnh 0.6 - 0.85 tùy ý
                    label = list(known_embeddings.keys())[np.argmax(sims)]
                    color = (0, 255, 0) # Màu xanh lá
                else:
                    label = "Unknown"
                    color = (0, 0, 255) # Màu đỏ
            else:
                label = "Unknown"
                color = (0, 0, 255)

            # Vẽ khung và text lên màn hình
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            display_text = f"{label}: {max_sim:.2f}" if label != "Unknown" else label
            cv2.putText(frame, display_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    cv2.imshow("Face Matching GPU", frame)
    if cv2.waitKey(1) & 0xFF == 27:  # Nhấn phím ESC để thoát
        break

cap.release()
cv2.destroyAllWindows()

d:\xu_li_anh\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sử dụng thiết bị: cuda
Đã load embeddings của: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '3', '4', '5', '6', '7', '8', '9', 'me']
